# 🏋️ VoC 客服文本分析 — 自我练习版

> 参考答案在 `39_VoC客服文本分析_分层高频词提取.ipynb`

## 核心流程 (不用 LDA！)
1. 全集统一跑 TF-IDF（保证 IDF 基准一致）
2. 按已有的 `category` 主题标签 GroupBy → 提取各主题高频词
3. 差评 - 好评 权重差 → 找出差评独有痛点词

---

## Step 0: 环境准备 & 数据导入（已完成）

In [1]:
# 数据下载（首次运行取消注释）
# !kaggle datasets download ddosad/ecommerce-customer-service-satisfaction --path ./data --unzip

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
else:
    plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')
print('✅ 环境配置完成！')

✅ 环境配置完成！


In [3]:
# 加载数据
df = pd.read_csv('./data/Customer_support_data.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(f'📊 数据集大小: {df.shape}')
print(f'📊 列名: {list(df.columns)}')
df.head(3)

📊 数据集大小: (85907, 20)
📊 列名: ['unique_id', 'channel_name', 'category', 'sub-category', 'customer_remarks', 'order_id', 'order_date_time', 'issue_reported_at', 'issue_responded', 'survey_response_date', 'customer_city', 'product_category', 'item_price', 'connected_handling_time', 'agent_name', 'supervisor', 'manager', 'tenure_bucket', 'agent_shift', 'csat_score']


,unique_id,channel_name,category,sub-category,customer_remarks,order_id,order_date_time,issue_reported_at,issue_responded,survey_response_date,customer_city,product_category,item_price,connected_handling_time,agent_name,supervisor,manager,tenure_bucket,agent_shift,csat_score
0,7e9ae164-6a8b-4521-a2d4-58f7c9fff13f,Outcall,Product Queries,Life Insurance,NaN,c27c9bb4-fa36-4140-9f1f-21009254ffdb,NaN,01/08/2023 11:13,01/08/2023 11:47,01-Aug-23,NaN,NaN,NaN,NaN,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5
1,b07ec1b0-f376-43b6-86df-ec03da3b2e16,Outcall,Product Queries,Product Specific Information,NaN,d406b0c7-ce17-4654-b9de-f08d421254bd,NaN,01/08/2023 12:52,01/08/2023 12:54,01-Aug-23,NaN,NaN,NaN,NaN,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5
2,200814dd-27c7-4149-ba2b-bd3af3092880,Inbound,Order Related,Installation/demo,NaN,c273368d-b961-44cb-beaf-62d6fd6c00d5,NaN,01/08/2023 20:16,01/08/2023 20:38,01-Aug-23,NaN,NaN,NaN,NaN,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5


---
## Step 1: 数据探索 & 理解业务语境
**任务**: 了解缺失值、主题标签分布、CSAT 评分分布，定义差评阈值

In [4]:
# TODO: 1.1 缺失值概览 — 关注 customer_remarks 有多少有文本
df.isnull().mean()

unique_id                  0.000000
channel_name               0.000000
category                   0.000000
sub-category               0.000000
customer_remarks           0.665429
order_id                   0.212230
order_date_time            0.799621
issue_reported_at          0.000000
issue_responded            0.000000
survey_response_date       0.000000
customer_city              0.801192
product_category           0.799830
item_price                 0.799714
connected_handling_time    0.997183
agent_name                 0.000000
supervisor                 0.000000
manager                    0.000000
tenure_bucket              0.000000
agent_shift                0.000000
csat_score                 0.000000
dtype: float64

In [5]:
# TODO: 1.2 主题标签分布 (category) — 条形图
df['category'].value_counts(normalize=True)

category
Returns               0.513311
Order Related         0.270234
Refund Related        0.052964
Product Queries       0.042977
Shopzilla Related     0.032500
Payments related      0.027087
Feedback              0.026703
Cancellation          0.025749
Offers & Cashback     0.005587
Others                0.001152
App/website           0.000978
Onboarding related    0.000757
Name: proportion, dtype: float64

In [6]:
df['csat_score'].info()

<class 'pandas.Series'>
RangeIndex: 85907 entries, 0 to 85906
Series name: csat_score
Non-Null Count  Dtype
--------------  -----
85907 non-null  int64
dtypes: int64(1)
memory usage: 671.3 KB


In [7]:
# TODO: 1.3 CSAT 评分分布 + 定义 NEGATIVE_THRESHOLD
NEGATIVE_THRESHOLD = 2  # CSAT <= 2 视为不满意
df = df.sort_values('csat_score',ascending=True)
df['csat_score'].value_counts(normalize=True).sort_index(ascending=True).cumsum()




csat_score
1    0.130723
2    0.145658
3    0.175434
4    0.306029
5    1.000000
Name: proportion, dtype: float64

In [8]:
# TODO: 1.4 各主题的差评率对比 — 用 groupby + agg
df['is_negative'] = (df['csat_score'] <= NEGATIVE_THRESHOLD).astype(int)
df.groupby('category')['is_negative'].mean()

category
App/website           0.130952
Cancellation          0.214738
Feedback              0.168265
Offers & Cashback     0.147917
Onboarding related    0.153846
Order Related         0.178850
Others                0.333333
Payments related      0.117318
Product Queries       0.182557
Refund Related        0.150110
Returns               0.121845
Shopzilla Related     0.133238
Name: is_negative, dtype: float64

---
## Step 2: 文本预处理
**任务**: 构建 `preprocess_text()` 函数，批量清洗 customer_remarks

In [9]:
# TODO: 2.1 构建预处理管道
# 提示: 小写 → re.sub 去标点 → word_tokenize → 去停用词 → lemmatize
# 别忘了添加业务自定义停用词 (order, product, item, please, thank 等)

lemmatizer = WordNetLemmatizer()     # 词形还原器 running > run
stop_words = set(stopwords.words('english')) # 加载英文停用词表

# 业务自定义停用词（客服场景高频但无分析价值的词）
CUSTOM_STOPWORDS = {
    'order', 'product', 'item', 'please', 'thank',
    'customer', 'service', 'support', 'help', 'want',
    'need', 'get', 'got', 'would', 'could', 'also',
    'one', 'like', 'good', 'great', 'well', 'really',
}
stop_words = stop_words.union(CUSTOM_STOPWORDS)

def processing_text(text:str) -> str:
    if pd.isna(text):
        return ' '
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens 
        if token.isalpha() and token not in stop_words and len(token) > 2
    ]
    return ' '.join(tokens)

print('预处理函数定义完成')



预处理函数定义完成


In [13]:
# TODO: 2.2 批量预处理 + 过滤无文本行
# 提示: dropna(subset=['customer_remarks']) → apply(preprocess_text) → 过滤空串
df_text = df.dropna(subset=['customer_remarks']).copy()
print(f'有文本记录：{len(df_text):,} / {len(df):,}')

df_text['clean_text'] = df_text['customer_remarks'].apply(processing_text)

df_text = df_text[df_text['clean_text'].str.strip() != ''].copy()
print(f'清洗后有效文本：{len(df_text):,}')
print(f'差评占比:{df_text['is_negative'].mean():.1%}')

df_text[['customer_remarks', 'clean_text', 'category', 'sub-category', 'csat_score']].head(5)


有文本记录：28,742 / 85,907
清洗后有效文本：21,630
差评占比:24.9%


,customer_remarks,clean_text,category,sub-category,csat_score
34224,Pathetic customer support,pathetic,Returns,Reverse Pickup Enquiry,1
25956,Package found empty recommend it fast,package found empty recommend fast,Returns,Missing,1
25953,Call back,call back,Order Related,Order status enquiry,1
45286,My order not pickup today Because when a callW...,pickup today callwhy refund canceled,Returns,Reverse Pickup Enquiry,1
11435,Realme very poor product,realme poor,Order Related,Installation/demo,1


---
## Step 3: 全集统一 TF-IDF → 按主题标签 GroupBy 提取高频词
**核心**: 不要分组各自跑 TF-IDF！统一跑一次，再按 category 做 GroupBy

In [14]:
# TODO: 3.1 全集统一跑 TF-IDF
TFIDF_MAX_FEATURES = 500
TFIDF_MIN_DF = 5
TFIDF_NGRAM_RANGE = (1,2)

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    ngram_range=TFIDF_NGRAM_RANGE
)
X_tfidf = tfidf.fit_transform(df_text['clean_text'])

print(f'TF-IDF shape:{X_tfidf.shape}')
print(f'size:{len(tfidf.get_feature_names_out())}')

TF-IDF shape:(21630, 500)
size:500


In [16]:
# TODO: 3.2 转成 DataFrame，附上 category 列
df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns = tfidf.get_feature_names_out(),
    index=df_text.index
)
df_tfidf['category'] = df_text['category'].values

print(f'TF-IDF Dataframe shape:{df_tfidf.shape}')
df_tfidf.head()

TF-IDF Dataframe shape:(21630, 501)


,aap,able,account,achha,action,actually,add,address,agent,already,...,word,work,working,worst,worst experience,wrong,year,yes,yet,category
34224,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Returns
25956,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Returns
25953,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Order Related
45286,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Returns
11435,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Order Related


In [ ]:
# TODO: 3.3 按主题 GroupBy → 每个主题取 Top 10 高权重词


In [ ]:
# TODO: 3.4 可视化：各主题 Top 关键词热力图


---
## Step 4: 差评 vs 好评 对比分析（按主题下钻）
**任务**: 对每个主题，计算差评和好评的平均 TF-IDF，取差值找差评独有痛点词

In [ ]:
# TODO: 4.1 选择 Returns 主题，对比差评 vs 好评 Top 关键词


In [ ]:
# TODO: 4.2 差评独有痛点词（差评权重 - 好评权重，差值最大的词）


In [ ]:
# TODO: 4.3 对所有主要主题自动化跑差评独有痛点词


---
## Step 5: Sub-category 二级下钻
**任务**: 在 Returns 主题内部，用 sub-category 做二级差评率分析

In [ ]:
# TODO: 5.1 Returns 下 Sub-category 差评率 + 可视化


---
## Step 6: 优先级评分模型
**任务**: 差评量 × 差评率 × 涉及渠道数 → 归一化 → 综合优先级 + 气泡图

In [ ]:
# TODO: 6.1 各主题优先级评分


In [ ]:
# TODO: 6.2 气泡图：差评率 × 差评量 × 渠道数


---
## 📋 结论与建议
**TODO**: 写出你的分析结论和行动建议